In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
    mean_squared_error
)

print("Residual U-Net SR Experiment")
print("PyTorch:", torch.__version__)

Residual U-Net SR Experiment
PyTorch: 2.5.1+cu121


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Quadro GV100


In [3]:
GT_PATH = "../data/train/GT"
NOISY_PATH = "../data/train/NoisyLR"

files = sorted([
    f for f in os.listdir(GT_PATH)
    if f.endswith(".npy")
])

print("Total pairs:", len(files))

Total pairs: 3200


In [4]:
np.random.seed(SEED)

indices = np.random.permutation(
    len(files)
)

train_end = int(
    0.80 * len(files)
)

val_end = int(
    0.90 * len(files)
)

train_files = [
    files[i]
    for i in indices[:train_end]
]

val_files = [
    files[i]
    for i in indices[
        train_end:val_end
    ]
]

test_files = [
    files[i]
    for i in indices[
        val_end:
    ]
]

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

Train: 2560
Validation: 320
Test: 320


In [5]:
class KLADataset(Dataset):

    def __init__(
        self,
        gt_path,
        noisy_path,
        files
    ):

        self.gt_path = gt_path
        self.noisy_path = noisy_path
        self.files = files


    def __len__(self):

        return len(self.files)


    def __getitem__(
        self,
        idx
    ):

        filename = self.files[idx]

        lr = np.load(
            os.path.join(
                self.noisy_path,
                filename
            )
        ).astype(
            np.float32
        )

        gt = np.load(
            os.path.join(
                self.gt_path,
                filename
            )
        ).astype(
            np.float32
        )

        # IMPORTANT:
        # no early clipping

        lr = torch.from_numpy(
            lr
        ).unsqueeze(0)

        gt = torch.from_numpy(
            gt
        ).unsqueeze(0)

        return lr, gt

In [6]:
BATCH_SIZE = 8

train_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    train_files
)

val_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    val_files
)

test_dataset = KLADataset(
    GT_PATH,
    NOISY_PATH,
    test_files
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)

2560 320 320


In [7]:
class ResidualBlock(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.act = nn.LeakyReLU(
            0.1,
            inplace=True
        )


    def forward(
        self,
        x
    ):

        residual = x

        x = self.conv1(
            x
        )

        x = self.act(
            x
        )

        x = self.conv2(
            x
        )

        return (
            residual
            +
            0.1 * x
        )

In [8]:
class EncoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.project = nn.Conv2d(
            in_channels,
            out_channels,
            3,
            padding=1
        )

        self.res1 = ResidualBlock(
            out_channels
        )

        self.res2 = ResidualBlock(
            out_channels
        )

        self.down = nn.Conv2d(
            out_channels,
            out_channels,
            3,
            stride=2,
            padding=1
        )


    def forward(
        self,
        x
    ):

        x = self.project(
            x
        )

        x = self.res1(
            x
        )

        x = self.res2(
            x
        )

        skip = x

        x = self.down(
            x
        )

        return x, skip

In [9]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels
    ):

        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.fuse = nn.Conv2d(
            out_channels + skip_channels,
            out_channels,
            3,
            padding=1
        )

        self.res1 = ResidualBlock(
            out_channels
        )

        self.res2 = ResidualBlock(
            out_channels
        )


    def forward(
        self,
        x,
        skip
    ):

        x = self.up(
            x
        )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.fuse(
            x
        )

        x = self.res1(
            x
        )

        x = self.res2(
            x
        )

        return x

In [10]:
class ResidualUNetSR(nn.Module):

    def __init__(
        self,
        base_channels=48,
        scale=2
    ):

        super().__init__()


        # Initial features
        self.head = nn.Conv2d(
            1,
            base_channels,
            3,
            padding=1
        )


        # Encoder
        self.enc1 = EncoderBlock(
            base_channels,
            base_channels
        )

        self.enc2 = EncoderBlock(
            base_channels,
            base_channels * 2
        )


        # Bottleneck
        self.bottleneck_in = nn.Conv2d(
            base_channels * 2,
            base_channels * 3,
            3,
            padding=1
        )

        self.bottleneck = nn.Sequential(
            ResidualBlock(
                base_channels * 3
            ),

            ResidualBlock(
                base_channels * 3
            ),

            ResidualBlock(
                base_channels * 3
            ),

            ResidualBlock(
                base_channels * 3
            )
        )


        # Decoder
        self.dec2 = DecoderBlock(
            base_channels * 3,
            base_channels * 2,
            base_channels * 2
        )

        self.dec1 = DecoderBlock(
            base_channels * 2,
            base_channels,
            base_channels
        )


        # Feature refinement
        self.refine = nn.Sequential(
            ResidualBlock(
                base_channels
            ),
            ResidualBlock(
                base_channels
            )
        )


        # Super-resolution head
        self.up = nn.Sequential(

            nn.Conv2d(
                base_channels,
                base_channels
                *
                scale
                *
                scale,
                3,
                padding=1
            ),

            nn.PixelShuffle(
                scale
            ),

            nn.Conv2d(
                base_channels,
                1,
                3,
                padding=1
            )
        )


    def forward(
        self,
        x
    ):

        x = self.head(
            x
        )

        shallow = x


        x, skip1 = self.enc1(
            x
        )

        x, skip2 = self.enc2(
            x
        )


        x = self.bottleneck_in(
            x
        )

        x = self.bottleneck(
            x
        )


        x = self.dec2(
            x,
            skip2
        )

        x = self.dec1(
            x,
            skip1
        )


        x = self.refine(
            x
        )


        # Long residual in feature space
        x = (
            x
            +
            shallow
        )


        output = self.up(
            x
        )


        return output

In [11]:
model = ResidualUNetSR(
    base_channels=48,
    scale=2
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)

print(
    "\nTotal parameters:",
    total_params
)

print(
    "Trainable parameters:",
    trainable_params
)

ResidualUNetSR(
  (head): Conv2d(1, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc1): EncoderBlock(
    (project): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (res1): ResidualBlock(
      (conv1): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (act): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (res2): ResidualBlock(
      (conv1): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (conv2): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (act): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (down): Conv2d(48, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
  (enc2): EncoderBlock(
    (project): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (res1): ResidualBlock(
      (conv1): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
     

In [12]:
dummy = torch.randn(
    2,
    1,
    128,
    128
).to(device)

with torch.no_grad():

    output = model(
        dummy
    )

print(
    "Input:",
    dummy.shape
)

print(
    "Output:",
    output.shape
)

Input: torch.Size([2, 1, 128, 128])
Output: torch.Size([2, 1, 256, 256])


In [13]:
def gaussian_window(
    window_size=11,
    sigma=1.5,
    channels=1,
    device="cpu"
):

    coords = torch.arange(
        window_size,
        dtype=torch.float32,
        device=device
    )

    coords = (
        coords
        -
        window_size // 2
    )

    g = torch.exp(
        -(
            coords ** 2
        )
        /
        (
            2 * sigma ** 2
        )
    )

    g = g / g.sum()

    window = (
        g[:, None]
        *
        g[None, :]
    )

    return window.expand(
        channels,
        1,
        window_size,
        window_size
    )

In [14]:
def differentiable_ssim(
    x,
    y,
    window_size=11
):

    channels = x.shape[1]

    window = gaussian_window(
        window_size,
        1.5,
        channels,
        x.device
    )

    mu_x = F.conv2d(
        x,
        window,
        padding=window_size//2,
        groups=channels
    )

    mu_y = F.conv2d(
        y,
        window,
        padding=window_size//2,
        groups=channels
    )

    mu_x2 = mu_x ** 2
    mu_y2 = mu_y ** 2
    mu_xy = mu_x * mu_y

    sigma_x = (
        F.conv2d(
            x*x,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu_x2
    )

    sigma_y = (
        F.conv2d(
            y*y,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu_y2
    )

    sigma_xy = (
        F.conv2d(
            x*y,
            window,
            padding=window_size//2,
            groups=channels
        )
        -
        mu_xy
    )

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    ssim_map = (
        (
            2 * mu_xy + C1
        )
        *
        (
            2 * sigma_xy + C2
        )
    ) / (
        (
            mu_x2 + mu_y2 + C1
        )
        *
        (
            sigma_x + sigma_y + C2
        )
        +
        1e-8
    )

    return ssim_map.mean()

In [15]:
def restoration_loss(
    output,
    target
):

    l1 = F.l1_loss(
        output,
        target
    )

    ssim_value = differentiable_ssim(
        output,
        target
    )

    ssim_loss = (
        1.0
        -
        ssim_value
    )

    total_loss = (
        l1
        +
        0.10 * ssim_loss
    )

    return (
        total_loss,
        l1,
        ssim_loss
    )

In [16]:
EPOCHS = 30

optimizer = optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

print("Optimizer ready")

Optimizer ready


In [17]:
MODEL_DIR = "../models/resunet_sr"
RESULT_DIR = "../results/resunet_sr"
RESTORED_DIR = "../results/resunet_sr/restored"

for path in [
    MODEL_DIR,
    RESULT_DIR,
    RESTORED_DIR
]:

    os.makedirs(
        path,
        exist_ok=True
    )

print("Directories ready")

Directories ready


In [18]:
def evaluate_validation(
    model,
    loader
):

    model.eval()

    psnr_values = []
    ssim_values = []

    with torch.no_grad():

        for lr, gt in loader:

            lr = lr.to(
                device,
                non_blocking=True
            )

            output = model(
                lr
            )

            output = torch.clamp(
                output,
                0,
                1
            )

            output_np = (
                output
                .cpu()
                .numpy()
            )

            gt_np = (
                gt.numpy()
            )

            for i in range(
                output_np.shape[0]
            ):

                pred = output_np[
                    i,
                    0
                ]

                target = gt_np[
                    i,
                    0
                ]

                psnr_values.append(
                    peak_signal_noise_ratio(
                        target,
                        pred,
                        data_range=1.0
                    )
                )

                ssim_values.append(
                    structural_similarity(
                        target,
                        pred,
                        data_range=1.0
                    )
                )

    return (
        np.mean(psnr_values),
        np.mean(ssim_values)
    )

In [19]:
best_psnr = -1.0

train_history = []
val_psnr_history = []
val_ssim_history = []


for epoch in range(
    EPOCHS
):

    # ==========================
    # TRAIN
    # ==========================

    model.train()

    running_loss = 0.0


    for lr, gt in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    ):

        lr = lr.to(
            device,
            non_blocking=True
        )

        gt = gt.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )


        output = model(
            lr
        )


        (
            loss,
            _,
            _
        ) = restoration_loss(
            output,
            gt
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )


        optimizer.step()


        running_loss += (
            loss.item()
        )


    train_loss = (
        running_loss
        /
        len(train_loader)
    )


    scheduler.step()


    # ==========================
    # VALIDATION
    # ==========================

    val_psnr, val_ssim = (
        evaluate_validation(
            model,
            val_loader
        )
    )


    train_history.append(
        train_loss
    )

    val_psnr_history.append(
        val_psnr
    )

    val_ssim_history.append(
        val_ssim
    )


    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Loss: {train_loss:.6f} | "
        f"PSNR: {val_psnr:.4f} dB | "
        f"SSIM: {val_ssim:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.7f}"
    )


    # ==========================
    # SAVE BEST
    # ==========================

    if val_psnr > best_psnr:

        best_psnr = val_psnr

        checkpoint = {

            "epoch":
                epoch + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "val_psnr":
                val_psnr,

            "val_ssim":
                val_ssim,

            "config": {
                "base_channels": 48,
                "scale": 2
            }
        }

        torch.save(
            checkpoint,
            "../models/resunet_sr/"
            "resunet_sr_best.pth"
        )

        print(
            "★ Best model saved"
        )

Epoch 1/30: 100%|██████████| 320/320 [00:33<00:00,  9.57it/s]


Epoch [01/30] Loss: 0.088583 | PSNR: 25.6451 dB | SSIM: 0.6392 | LR: 0.0001995
★ Best model saved


Epoch 2/30: 100%|██████████| 320/320 [00:32<00:00,  9.77it/s]


Epoch [02/30] Loss: 0.070770 | PSNR: 26.4171 dB | SSIM: 0.6864 | LR: 0.0001978
★ Best model saved


Epoch 3/30: 100%|██████████| 320/320 [00:33<00:00,  9.59it/s]


...

Epoch [28/30] Loss: 0.053210 | PSNR: 28.2357 dB | SSIM: 0.7676 | LR: 0.0000032


Epoch 29/30: 100%|██████████| 320/320 [00:32<00:00,  9.95it/s]


Epoch [29/30] Loss: 0.053181 | PSNR: 28.2474 dB | SSIM: 0.7673 | LR: 0.0000015


Epoch 30/30: 100%|██████████| 320/320 [00:32<00:00,  9.97it/s]


Epoch [30/30] Loss: 0.053166 | PSNR: 28.2454 dB | SSIM: 0.7677 | LR: 0.0000010


In [20]:
checkpoint = torch.load(
    "../models/resunet_sr/resunet_sr_best.pth",
    map_location=device
)

best_model = ResidualUNetSR(
    base_channels=checkpoint["config"]["base_channels"],
    scale=checkpoint["config"]["scale"]
).to(device)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()

print("Best epoch:", checkpoint["epoch"])
print("Validation PSNR:", checkpoint["val_psnr"])
print("Validation SSIM:", checkpoint["val_ssim"])

Best epoch: 27
Validation PSNR: 28.25228253978496
Validation SSIM: 0.7668456649812425


/tmp/ipykernel_3576940/1740630939.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [21]:
psnr_scores = []
ssim_scores = []
mse_scores = []
inference_times = []

best_model.eval()

with torch.no_grad():

    for i, (lr, gt) in enumerate(
        tqdm(
            test_loader,
            desc="Testing Residual U-Net SR"
        )
    ):

        lr = lr.to(
            device,
            non_blocking=True
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        output = best_model(
            lr
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_ms = (
            time.perf_counter()
            -
            start
        ) * 1000

        output = torch.clamp(
            output,
            0,
            1
        )

        restored = (
            output
            .squeeze()
            .cpu()
            .numpy()
        )

        gt_np = (
            gt
            .squeeze()
            .numpy()
        )

        psnr_scores.append(
            peak_signal_noise_ratio(
                gt_np,
                restored,
                data_range=1.0
            )
        )

        ssim_scores.append(
            structural_similarity(
                gt_np,
                restored,
                data_range=1.0
            )
        )

        mse_scores.append(
            mean_squared_error(
                gt_np,
                restored
            )
        )

        inference_times.append(
            elapsed_ms
        )

        filename = test_files[i]

        np.save(
            os.path.join(
                RESTORED_DIR,
                filename
            ),
            restored.astype(
                np.float32
            )
        )

print("Testing complete")

Testing Residual U-Net SR: 100%|██████████| 320/320 [00:03<00:00, 82.06it/s]

Testing complete


In [22]:
avg_psnr = np.mean(psnr_scores)
avg_ssim = np.mean(ssim_scores)
avg_mse = np.mean(mse_scores)
avg_time = np.mean(inference_times)

print(
    "========== RESIDUAL U-NET SR RESULTS =========="
)

print(
    f"Average PSNR : {avg_psnr:.6f} dB"
)

print(
    f"Average SSIM : {avg_ssim:.6f}"
)

print(
    f"Average MSE  : {avg_mse:.8f}"
)

print(
    f"Average inference time : "
    f"{avg_time:.3f} ms/image"
)

========== RESIDUAL U-NET SR RESULTS ==========
Average PSNR : 28.776597 dB
Average SSIM : 0.772058
Average MSE  : 0.00226146
Average inference time : 4.556 ms/image


In [23]:
comparison_df = pd.DataFrame({

    "Method": [
        "Bicubic",
        "SRCNN",
        "DnCNN",
        "EDSR",
        "NAF-SR",
        "Residual U-Net SR"
    ],

    "PSNR_dB": [
        22.478415628140233,
        27.52943205674495,
        27.893001035080903,
        27.986647229958713,
        27.840080,
        avg_psnr
    ],

    "SSIM": [
        0.5116624465067319,
        0.7250710888767316,
        0.7249596254696442,
        0.7425061760836545,
        0.743259,
        avg_ssim
    ],

    "MSE": [
        0.007338321273890881,
        0.0027344456903774946,
        0.0025748269964531107,
        0.0026003361484997337,
        0.00266457,
        avg_mse
    ]
})

display(comparison_df)

,Method,PSNR_dB,SSIM,MSE
0,Bicubic,22.478416,0.511662,0.007338
1,SRCNN,27.529432,0.725071,0.002734
2,DnCNN,27.893001,0.724960,0.002575
3,EDSR,27.986647,0.742506,0.002600
4,NAF-SR,27.840080,0.743259,0.002665
5,Residual U-Net SR,28.776597,0.772058,0.002261


In [24]:
metrics_df = pd.DataFrame({
    "Filename": test_files,
    "PSNR_dB": psnr_scores,
    "SSIM": ssim_scores,
    "MSE": mse_scores,
    "Inference_Time_ms": inference_times
})

metrics_df.to_csv(
    "../results/resunet_sr/per_image_metrics.csv",
    index=False
)

print("========== PERFORMANCE DISTRIBUTION ==========")

print("\nPSNR")
print(metrics_df["PSNR_dB"].describe())

print("\nSSIM")
print(metrics_df["SSIM"].describe())

print("\nMSE")
print(metrics_df["MSE"].describe())

print("\n========== FAILURE COUNTS ==========")

print(
    "PSNR < 25 dB:",
    (metrics_df["PSNR_dB"] < 25).sum()
)

print(
    "PSNR < 27 dB:",
    (metrics_df["PSNR_dB"] < 27).sum()
)

print(
    "SSIM < 0.60:",
    (metrics_df["SSIM"] < 0.60).sum()
)

print(
    "SSIM < 0.70:",
    (metrics_df["SSIM"] < 0.70).sum()
)

print(
    "SSIM > 0.85:",
    (metrics_df["SSIM"] > 0.85).sum()
)

print("\n========== WORST 20 PSNR ==========")

display(
    metrics_df
    .sort_values("PSNR_dB")
    .head(20)
)

print("\n========== WORST 20 SSIM ==========")

display(
    metrics_df
    .sort_values("SSIM")
    .head(20)
)

========== PERFORMANCE DISTRIBUTION ==========

PSNR
count    320.000000
mean      28.776597
std        4.418154
min       11.257683
25%       25.361851
50%       28.692394
75%       31.767401
max       39.887276
Name: PSNR_dB, dtype: float64

SSIM
count    320.000000
mean       0.772058
std        0.145673
min        0.299102
25%        0.712976
50%        0.805099
75%        0.875509
max        0.974011
Name: SSIM, dtype: float64

MSE
count    320.000000
mean       0.002261
std        0.004534
min        0.000103
25%        0.000666
50%        0.001351
75%        0.002909
max        0.074857
Name: MSE, dtype: float64

========== FAILURE COUNTS ==========
PSNR < 25 dB: 70
PSNR < 27 dB: 121
SSIM < 0.60: 38
SSIM < 0.70: 75
SSIM > 0.85: 107

========== WORST 20 PSNR ==========


,Filename,PSNR_dB,SSIM,MSE,Inference_Time_ms
28,000627.npy,11.257683,0.316511,0.074857,5.916462
262,002041.npy,17.651527,0.555479,0.017173,6.019659
234,002317.npy,19.228380,0.619123,0.011944,3.386581
46,002393.npy,20.872939,0.726441,0.008179,3.490604
105,000641.npy,20.927843,0.384170,0.008076,6.193485
189,000804.npy,21.358599,0.666106,0.007314,3.653991
297,001215.npy,21.469956,0.782365,0.007129,3.357182
172,000206.npy,21.498555,0.731002,0.007082,3.460385
220,000379.npy,21.629004,0.769557,0.006872,3.551041
202,000378.npy,21.714424,0.751691,0.006738,3.558130



========== WORST 20 SSIM ==========


,Filename,PSNR_dB,SSIM,MSE,Inference_Time_ms
211,002695.npy,23.085050,0.299102,0.004915,3.451256
175,000870.npy,24.629599,0.309735,0.003444,3.649728
258,001363.npy,24.088952,0.315426,0.003900,3.462306
28,000627.npy,11.257683,0.316511,0.074857,5.916462
222,000763.npy,23.484657,0.317608,0.004483,3.417519
176,000397.npy,25.343275,0.323569,0.002922,6.237299
58,002385.npy,22.546564,0.328955,0.005563,3.456239
85,000293.npy,26.103100,0.345383,0.002453,3.445776
311,000466.npy,27.139596,0.354521,0.001932,7.668601
137,002038.npy,25.420140,0.368167,0.002871,3.182997
